# 22. Robotics action policies — full-depth, small-tensor implementations

The models below reduce **tensor scale only** (width, image resolution, sequence length, action horizon, batch size) while preserving the demonstrated architecture depth/topology.

- ACT: ResNet-18 `[2,2,2,2]`, 4-layer CVAE encoder, 4-layer observation encoder, 7-layer action-query decoder.
- Diffusion Policy: three temporal U-Net resolutions, two conditional residual blocks per down/up stage, two middle blocks.
- pi0: So400m-style 27-layer vision Transformer, 18-layer VLM/action-expert joint stack, GQA, RoPE, exact prefix/state/action attention blocks, flow matching.
- FAST: DCT -> quantization -> BPE -> BPE decode -> inverse DCT.


In [ ]:
import math
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(12)
torch.set_num_threads(min(2, torch.get_num_threads()))
device = torch.device("cpu")
print("device:", device)


## 1. ACT


In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, input_channels, output_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            input_channels, output_channels, 3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(output_channels)
        self.conv2 = nn.Conv2d(
            output_channels, output_channels, 3, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(output_channels)

        if stride == 1 and input_channels == output_channels:
            self.skip = nn.Identity()
        else:
            self.skip = nn.Sequential(
                nn.Conv2d(
                    input_channels, output_channels, 1, stride=stride, bias=False
                ),
                nn.BatchNorm2d(output_channels),
            )

    def forward(self, x):
        hidden = F.relu(self.bn1(self.conv1(x)))
        hidden = self.bn2(self.conv2(hidden))
        return F.relu(hidden + self.skip(x))


class SmallWidthResNet18(nn.Module):
    def __init__(self, output_channels=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 8, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.MaxPool2d(3, stride=2, padding=1),
        )

        stage_widths = [8, 12, 16, 24]
        stages = []
        input_channels = 8
        for stage_index, stage_width in enumerate(stage_widths):
            stride = 1 if stage_index == 0 else 2
            stages.append(
                nn.Sequential(
                    BasicBlock(input_channels, stage_width, stride),
                    BasicBlock(stage_width, stage_width),
                )
            )
            input_channels = stage_width

        self.stages = nn.ModuleList(stages)
        self.projection = nn.Conv2d(input_channels, output_channels, 1)

    def forward(self, image):
        hidden = self.stem(image)
        for stage in self.stages:
            hidden = stage(hidden)
        return self.projection(hidden)


class SmallTensorACT(nn.Module):
    def __init__(self, action_dim=3, hidden_dim=32, chunk_size=4, latent_dim=8):
        super().__init__()
        self.latent_dim = latent_dim
        self.backbone = SmallWidthResNet18(hidden_dim)

        self.posterior_cls = nn.Parameter(torch.zeros(1, 1, hidden_dim))
        self.posterior_qpos = nn.Linear(action_dim, hidden_dim)
        self.posterior_action = nn.Linear(action_dim, hidden_dim)
        self.posterior_position = nn.Parameter(
            torch.randn(1, chunk_size + 2, hidden_dim) * 0.02
        )

        posterior_layer = nn.TransformerEncoderLayer(
            hidden_dim, 4, 2 * hidden_dim, batch_first=True
        )
        self.posterior_encoder = nn.TransformerEncoder(posterior_layer, 4)
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)

        self.latent_projection = nn.Linear(latent_dim, hidden_dim)
        self.qpos_projection = nn.Linear(action_dim, hidden_dim)
        self.additional_position = nn.Parameter(
            torch.randn(1, 2, hidden_dim) * 0.02
        )

        encoder_layer = nn.TransformerEncoderLayer(
            hidden_dim, 4, 2 * hidden_dim, batch_first=True
        )
        self.observation_encoder = nn.TransformerEncoder(encoder_layer, 4)

        decoder_layer = nn.TransformerDecoderLayer(
            hidden_dim, 4, 2 * hidden_dim, batch_first=True
        )
        self.action_decoder = nn.TransformerDecoder(decoder_layer, 7)
        self.action_queries = nn.Parameter(
            torch.randn(1, chunk_size, hidden_dim) * 0.02
        )
        self.action_head = nn.Linear(hidden_dim, action_dim)

    def encode_posterior(self, qpos, target_actions):
        batch_size = qpos.size(0)
        tokens = torch.cat(
            [
                self.posterior_cls.expand(batch_size, -1, -1),
                self.posterior_qpos(qpos).unsqueeze(1),
                self.posterior_action(target_actions),
            ],
            dim=1,
        )
        hidden = self.posterior_encoder(tokens + self.posterior_position)[:, 0]
        mu = self.mu(hidden)
        logvar = self.logvar(hidden)
        latent = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return latent, mu, logvar

    def forward(self, image, qpos, target_actions=None):
        batch_size = image.size(0)
        if target_actions is None:
            latent = torch.zeros(batch_size, self.latent_dim, device=image.device)
            mu = torch.zeros_like(latent)
            logvar = torch.zeros_like(latent)
        else:
            latent, mu, logvar = self.encode_posterior(qpos, target_actions)

        image_tokens = self.backbone(image).flatten(2).transpose(1, 2)
        memory = torch.cat(
            [
                self.latent_projection(latent).unsqueeze(1),
                self.qpos_projection(qpos).unsqueeze(1),
                image_tokens,
            ],
            dim=1,
        )
        memory_position = torch.cat(
            [
                self.additional_position.expand(batch_size, -1, -1),
                torch.zeros_like(image_tokens),
            ],
            dim=1,
        )
        memory = self.observation_encoder(memory + memory_position)
        queries = self.action_queries.expand(batch_size, -1, -1)
        decoded = self.action_decoder(queries, memory)
        return self.action_head(decoded), mu, logvar


act = SmallTensorACT().to(device)
act_image = torch.randn(2, 3, 32, 32, device=device)
act_qpos = torch.randn(2, 3, device=device)
act_target = torch.randn(2, 4, 3, device=device)
act_optimizer = torch.optim.AdamW(act.parameters(), lr=3e-4)

act_loss_history = []
for step in range(5):
    torch.manual_seed(123)
    act_optimizer.zero_grad()
    predicted, mu, logvar = act(act_image, act_qpos, act_target)
    reconstruction = F.l1_loss(predicted, act_target)
    kl = -0.5 * (1 + logvar - mu.square() - logvar.exp()).mean()
    loss = reconstruction + 0.005 * kl
    loss.backward()
    act_optimizer.step()

    act_loss_history.append(loss.item())
    print(f"ACT step {step + 1}: loss={loss.item():.6f}")

act.eval()
with torch.no_grad():
    act_inference, _, _ = act(act_image[:1], act_qpos[:1])
print("ACT loss history:", act_loss_history)
print("ACT inference shape:", act_inference.shape)


## 2. Diffusion Policy


In [ ]:
def diffusion_time_embedding(t, dim):
    half = dim // 2
    frequencies = torch.exp(
        -math.log(10000.0)
        * torch.arange(half, device=t.device)
        / max(half - 1, 1)
    )
    angles = t[:, None] * frequencies[None]
    return torch.cat([angles.sin(), angles.cos()], dim=-1)


class ConditionalResidual1D(nn.Module):
    def __init__(self, input_channels, output_channels, condition_dim, kernel_size=5):
        super().__init__()
        groups = min(4, output_channels)
        self.conv1 = nn.Conv1d(
            input_channels, output_channels, kernel_size, padding=kernel_size // 2
        )
        self.norm1 = nn.GroupNorm(groups, output_channels)
        self.conv2 = nn.Conv1d(
            output_channels, output_channels, kernel_size, padding=kernel_size // 2
        )
        self.norm2 = nn.GroupNorm(groups, output_channels)
        self.condition = nn.Linear(condition_dim, 2 * output_channels)
        self.skip = (
            nn.Identity()
            if input_channels == output_channels
            else nn.Conv1d(input_channels, output_channels, 1)
        )

    def forward(self, x, condition):
        hidden = F.mish(self.norm1(self.conv1(x)))
        scale, shift = self.condition(condition).chunk(2, dim=-1)
        hidden = hidden * (1 + scale[:, :, None]) + shift[:, :, None]
        hidden = F.mish(self.norm2(self.conv2(hidden)))
        return hidden + self.skip(x)


class DownStage(nn.Module):
    def __init__(self, input_channels, output_channels, condition_dim, downsample):
        super().__init__()
        self.block1 = ConditionalResidual1D(
            input_channels, output_channels, condition_dim
        )
        self.block2 = ConditionalResidual1D(
            output_channels, output_channels, condition_dim
        )
        self.downsample = (
            nn.Conv1d(output_channels, output_channels, 3, stride=2, padding=1)
            if downsample
            else nn.Identity()
        )

    def forward(self, x, condition):
        x = self.block1(x, condition)
        x = self.block2(x, condition)
        return x, self.downsample(x)


class UpStage(nn.Module):
    def __init__(
        self, input_channels, skip_channels, output_channels, condition_dim, upsample
    ):
        super().__init__()
        self.block1 = ConditionalResidual1D(
            input_channels + skip_channels, output_channels, condition_dim
        )
        self.block2 = ConditionalResidual1D(
            output_channels, output_channels, condition_dim
        )
        self.upsample = (
            nn.ConvTranspose1d(
                output_channels, output_channels, 4, stride=2, padding=1
            )
            if upsample
            else nn.Identity()
        )

    def forward(self, x, skip, condition):
        if x.size(-1) != skip.size(-1):
            x = F.interpolate(
                x, size=skip.size(-1), mode="linear", align_corners=False
            )
        x = torch.cat([x, skip], dim=1)
        x = self.block1(x, condition)
        x = self.block2(x, condition)
        return self.upsample(x)


class SmallTensorDiffusionPolicy(nn.Module):
    def __init__(
        self, observation_dim=8, action_dim=3, down_dims=(8, 16, 32), condition_dim=32
    ):
        super().__init__()
        self.condition_dim = condition_dim
        self.observation = nn.Linear(observation_dim, condition_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(condition_dim, condition_dim),
            nn.Mish(),
            nn.Linear(condition_dim, condition_dim),
        )
        self.input_projection = nn.Conv1d(action_dim, down_dims[0], 1)

        down_stages = []
        current_channels = down_dims[0]
        for stage_index, output_channels in enumerate(down_dims):
            down_stages.append(
                DownStage(
                    current_channels,
                    output_channels,
                    condition_dim,
                    downsample=stage_index < len(down_dims) - 1,
                )
            )
            current_channels = output_channels
        self.down_stages = nn.ModuleList(down_stages)

        self.middle1 = ConditionalResidual1D(
            down_dims[-1], down_dims[-1], condition_dim
        )
        self.middle2 = ConditionalResidual1D(
            down_dims[-1], down_dims[-1], condition_dim
        )

        up_stages = []
        current_channels = down_dims[-1]
        for stage_index in reversed(range(len(down_dims) - 1)):
            output_channels = down_dims[stage_index]
            up_stages.append(
                UpStage(
                    current_channels,
                    down_dims[stage_index],
                    output_channels,
                    condition_dim,
                    upsample=stage_index > 0,
                )
            )
            current_channels = output_channels
        self.up_stages = nn.ModuleList(up_stages)

        self.output = nn.Sequential(
            nn.Conv1d(down_dims[0], down_dims[0], 3, padding=1),
            nn.Mish(),
            nn.Conv1d(down_dims[0], action_dim, 1),
        )

    def forward(self, noisy_actions, observation, t):
        condition = self.observation(observation) + self.time_mlp(
            diffusion_time_embedding(t, self.condition_dim)
        )
        hidden = self.input_projection(noisy_actions.transpose(1, 2))

        skips = []
        for down_stage in self.down_stages:
            skip, hidden = down_stage(hidden, condition)
            skips.append(skip)

        hidden = self.middle1(hidden, condition)
        hidden = self.middle2(hidden, condition)
        for up_stage, skip in zip(self.up_stages, reversed(skips[:-1])):
            hidden = up_stage(hidden, skip, condition)

        return self.output(hidden).transpose(1, 2)


@torch.no_grad()
def sample_diffusion_policy(model, observation, horizon=8, action_dim=3, steps=6):
    actions = torch.randn(
        observation.size(0), horizon, action_dim, device=observation.device
    )
    for step in range(steps, 0, -1):
        current_t = torch.full(
            (observation.size(0),), step / steps, device=observation.device
        )
        next_t = torch.full(
            (observation.size(0),), (step - 1) / steps, device=observation.device
        )
        predicted_noise = model(actions, observation, current_t)
        alpha_now = torch.cos(0.5 * math.pi * current_t)[:, None, None]
        sigma_now = torch.sin(0.5 * math.pi * current_t)[:, None, None]
        x0_prediction = (
            actions - sigma_now * predicted_noise
        ) / alpha_now.clamp_min(1e-3)
        alpha_next = torch.cos(0.5 * math.pi * next_t)[:, None, None]
        sigma_next = torch.sin(0.5 * math.pi * next_t)[:, None, None]
        actions = alpha_next * x0_prediction + sigma_next * predicted_noise
    return actions


diffusion_policy = SmallTensorDiffusionPolicy().to(device)
dp_observation = torch.randn(2, 8, device=device)
dp_clean = torch.randn(2, 8, 3, device=device)
dp_noise = torch.randn_like(dp_clean)
dp_t = torch.tensor([0.3, 0.7], device=device)
dp_alpha = torch.cos(0.5 * math.pi * dp_t)[:, None, None]
dp_sigma = torch.sin(0.5 * math.pi * dp_t)[:, None, None]
dp_noisy = dp_alpha * dp_clean + dp_sigma * dp_noise
dp_optimizer = torch.optim.AdamW(diffusion_policy.parameters(), lr=2e-3)

dp_loss_history = []
for step in range(5):
    dp_optimizer.zero_grad()
    predicted_noise = diffusion_policy(dp_noisy, dp_observation, dp_t)
    loss = F.mse_loss(predicted_noise, dp_noise)
    loss.backward()
    dp_optimizer.step()
    dp_loss_history.append(loss.item())
    print(f"Diffusion Policy step {step + 1}: loss={loss.item():.6f}")

print("Diffusion Policy loss history:", dp_loss_history)
print(
    "Diffusion Policy rollout:",
    sample_diffusion_policy(diffusion_policy, dp_observation[:1]).shape,
)


## 3. pi0

The suffix mask follows openpi's block construction exactly: prefix tokens form one full-attention block; the state token starts a new block and therefore cannot see action tokens; the first action token starts the action block, and all action tokens share that block.


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        scale = torch.rsqrt(x.square().mean(dim=-1, keepdim=True) + self.eps)
        return x * scale * self.weight


class VisionTransformerBlock(nn.Module):
    def __init__(self, dim=16, heads=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attention = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim),
        )

    def forward(self, x):
        normalized = self.norm1(x)
        attended, _ = self.attention(
            normalized, normalized, normalized, need_weights=False
        )
        x = x + attended
        return x + self.mlp(self.norm2(x))


class SmallTensorSigLIP(nn.Module):
    def __init__(self, dim=16, depth=27, image_size=8, patch_size=4):
        super().__init__()
        self.patch = nn.Conv2d(3, dim, patch_size, stride=patch_size)
        patch_count = (image_size // patch_size) ** 2
        self.position = nn.Parameter(torch.randn(1, patch_count, dim) * 0.02)
        self.blocks = nn.ModuleList(
            [VisionTransformerBlock(dim) for _ in range(depth)]
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, image):
        tokens = self.patch(image).flatten(2).transpose(1, 2) + self.position
        for block in self.blocks:
            tokens = block(tokens)
        return self.norm(tokens)


def rotate_half(x):
    even = x[..., 0::2]
    odd = x[..., 1::2]
    return torch.stack([-odd, even], dim=-1).flatten(-2)


def apply_rope(x, positions):
    head_dim = x.size(-1)
    frequencies = 1.0 / (
        10000.0
        ** (
            torch.arange(0, head_dim, 2, device=x.device, dtype=x.dtype)
            / head_dim
        )
    )
    angles = positions.to(x.dtype)[None, None, :, None] * frequencies[None, None, None, :]
    cosine = torch.repeat_interleave(angles.cos(), 2, dim=-1)
    sine = torch.repeat_interleave(angles.sin(), 2, dim=-1)
    return x * cosine + rotate_half(x) * sine


class DualStreamGemmaBlock(nn.Module):
    def __init__(self, dim=16, query_heads=4, kv_heads=1):
        super().__init__()
        assert dim % query_heads == 0
        assert query_heads % kv_heads == 0
        self.query_heads = query_heads
        self.kv_heads = kv_heads
        self.head_dim = dim // query_heads

        self.prefix_norm = RMSNorm(dim)
        self.expert_norm = RMSNorm(dim)
        self.prefix_q = nn.Linear(dim, query_heads * self.head_dim, bias=False)
        self.prefix_k = nn.Linear(dim, kv_heads * self.head_dim, bias=False)
        self.prefix_v = nn.Linear(dim, kv_heads * self.head_dim, bias=False)
        self.prefix_out = nn.Linear(query_heads * self.head_dim, dim, bias=False)
        self.expert_q = nn.Linear(dim, query_heads * self.head_dim, bias=False)
        self.expert_k = nn.Linear(dim, kv_heads * self.head_dim, bias=False)
        self.expert_v = nn.Linear(dim, kv_heads * self.head_dim, bias=False)
        self.expert_out = nn.Linear(query_heads * self.head_dim, dim, bias=False)

        self.prefix_ffn_norm = RMSNorm(dim)
        self.expert_ffn_norm = RMSNorm(dim)
        self.prefix_ffn = self._make_ffn(dim)
        self.expert_ffn = self._make_ffn(dim)

    @staticmethod
    def _make_ffn(dim):
        return nn.ModuleDict(
            {
                "gate": nn.Linear(dim, 4 * dim, bias=False),
                "up": nn.Linear(dim, 4 * dim, bias=False),
                "down": nn.Linear(4 * dim, dim, bias=False),
            }
        )

    def _project(self, x, q_projection, k_projection, v_projection):
        batch, length, _ = x.shape
        q = q_projection(x).view(
            batch, length, self.query_heads, self.head_dim
        ).transpose(1, 2)
        k = k_projection(x).view(
            batch, length, self.kv_heads, self.head_dim
        ).transpose(1, 2)
        v = v_projection(x).view(
            batch, length, self.kv_heads, self.head_dim
        ).transpose(1, 2)
        return q, k, v

    @staticmethod
    def _ffn(x, module):
        gate = F.gelu(module["gate"](x), approximate="tanh")
        value = module["up"](x)
        return module["down"](gate * value)

    def forward(self, prefix, expert, attention_mask):
        prefix_length = prefix.size(1)
        expert_length = expert.size(1)
        prefix_normalized = self.prefix_norm(prefix)
        expert_normalized = self.expert_norm(expert)

        prefix_q, prefix_k, prefix_v = self._project(
            prefix_normalized, self.prefix_q, self.prefix_k, self.prefix_v
        )
        expert_q, expert_k, expert_v = self._project(
            expert_normalized, self.expert_q, self.expert_k, self.expert_v
        )

        positions = torch.arange(
            prefix_length + expert_length, device=prefix.device
        )
        prefix_q = apply_rope(prefix_q, positions[:prefix_length])
        prefix_k = apply_rope(prefix_k, positions[:prefix_length])
        expert_q = apply_rope(expert_q, positions[prefix_length:])
        expert_k = apply_rope(expert_k, positions[prefix_length:])

        repeats = self.query_heads // self.kv_heads
        prefix_k = prefix_k.repeat_interleave(repeats, dim=1)
        prefix_v = prefix_v.repeat_interleave(repeats, dim=1)
        expert_k = expert_k.repeat_interleave(repeats, dim=1)
        expert_v = expert_v.repeat_interleave(repeats, dim=1)

        q = torch.cat([prefix_q, expert_q], dim=2)
        k = torch.cat([prefix_k, expert_k], dim=2)
        v = torch.cat([prefix_v, expert_v], dim=2)
        attended = F.scaled_dot_product_attention(
            q, k, v, attn_mask=attention_mask[None, None]
        )

        prefix_attended = (
            attended[:, :, :prefix_length].transpose(1, 2).contiguous().flatten(2)
        )
        expert_attended = (
            attended[:, :, prefix_length:].transpose(1, 2).contiguous().flatten(2)
        )
        prefix = prefix + self.prefix_out(prefix_attended)
        expert = expert + self.expert_out(expert_attended)
        prefix = prefix + self._ffn(
            self.prefix_ffn_norm(prefix), self.prefix_ffn
        )
        expert = expert + self._ffn(
            self.expert_ffn_norm(expert), self.expert_ffn
        )
        return prefix, expert


class SmallTensorPi0(nn.Module):
    def __init__(self, dim=16, action_dim=3, horizon=4, vocab_size=32):
        super().__init__()
        self.dim = dim
        self.action_dim = action_dim
        self.horizon = horizon
        self.vision_encoder = SmallTensorSigLIP(dim=dim, depth=27)
        self.language_embedding = nn.Embedding(vocab_size, dim)
        self.state_projection = nn.Linear(action_dim, dim)
        self.action_projection = nn.Linear(action_dim, dim)
        self.action_time_mlp_in = nn.Linear(2 * dim, dim)
        self.action_time_mlp_out = nn.Linear(dim, dim)
        self.joint_blocks = nn.ModuleList(
            [DualStreamGemmaBlock(dim) for _ in range(18)]
        )
        self.action_out = nn.Linear(dim, action_dim)

    def _time_embedding(self, t):
        half = self.dim // 2
        fraction = torch.linspace(0.0, 1.0, half, device=t.device)
        periods = 4e-3 * (4.0 / 4e-3) ** fraction
        angles = t[:, None] / periods[None] * 2 * math.pi
        return torch.cat([angles.sin(), angles.cos()], dim=-1)

    @staticmethod
    def _joint_mask(prefix_length, expert_length, device):
        if expert_length < 2:
            raise ValueError("pi0 suffix requires state + action tokens")

        total = prefix_length + expert_length
        mask = torch.zeros(total, total, dtype=torch.bool, device=device)

        # Prefix image/language block: full attention only inside prefix.
        mask[:prefix_length, :prefix_length] = True

        # State starts a new block: it can read prefix + itself, not actions.
        state_index = prefix_length
        mask[state_index, : state_index + 1] = True

        # The first action starts another block. All actions share that block.
        action_start = state_index + 1
        mask[action_start:, :] = True
        return mask

    def forward(self, images, language, state, noisy_actions, t):
        image_tokens = [
            self.vision_encoder(images[:, camera_index])
            for camera_index in range(images.size(1))
        ]
        prefix = torch.cat(
            image_tokens + [self.language_embedding(language)], dim=1
        )

        time_embedding = self._time_embedding(t)
        action_tokens = self.action_projection(noisy_actions)
        expanded_time = time_embedding[:, None].expand(-1, self.horizon, -1)
        action_tokens = torch.cat([action_tokens, expanded_time], dim=-1)
        action_tokens = self.action_time_mlp_out(
            F.silu(self.action_time_mlp_in(action_tokens))
        )
        expert = torch.cat(
            [self.state_projection(state).unsqueeze(1), action_tokens], dim=1
        )

        attention_mask = self._joint_mask(
            prefix.size(1), expert.size(1), prefix.device
        )
        for block in self.joint_blocks:
            prefix, expert = block(prefix, expert, attention_mask)
        return self.action_out(expert[:, -self.horizon :])


@torch.no_grad()
def sample_pi0(model, images, language, state, steps=6):
    actions = torch.randn(
        state.size(0), model.horizon, model.action_dim, device=state.device
    )
    for step in range(steps, 0, -1):
        t = torch.full(
            (state.size(0),), step / steps, device=state.device
        )
        velocity = model(images, language, state, actions, t)
        actions = actions - velocity / steps
    return actions


pi0 = SmallTensorPi0().to(device)
pi0_images = torch.randn(1, 3, 3, 8, 8, device=device)
pi0_language = torch.randint(0, 32, (1, 3), device=device)
pi0_state = torch.randn(1, 3, device=device)
pi0_actions = torch.randn(1, 4, 3, device=device)
pi0_noise = torch.randn_like(pi0_actions)
pi0_t = torch.tensor([0.6], device=device)
pi0_noisy = (
    (1 - pi0_t[:, None, None]) * pi0_actions
    + pi0_t[:, None, None] * pi0_noise
)
pi0_target_velocity = pi0_noise - pi0_actions
pi0_optimizer = torch.optim.AdamW(pi0.parameters(), lr=5e-4)

pi0_loss_history = []
for step in range(5):
    pi0_optimizer.zero_grad()
    predicted_velocity = pi0(
        pi0_images, pi0_language, pi0_state, pi0_noisy, pi0_t
    )
    loss = F.mse_loss(predicted_velocity, pi0_target_velocity)
    loss.backward()
    pi0_optimizer.step()
    pi0_loss_history.append(loss.item())
    print(f"pi0 step {step + 1}: loss={loss.item():.6f}")

print("pi0 loss history:", pi0_loss_history)
print(
    "pi0 rollout:",
    sample_pi0(pi0, pi0_images, pi0_language, pi0_state).shape,
)


## 4. FAST tokenizer round trip


In [ ]:
def dct_matrix(length, device):
    sample_index = torch.arange(length, device=device, dtype=torch.float32)
    frequency_index = torch.arange(
        length, device=device, dtype=torch.float32
    )[:, None]
    matrix = torch.cos(
        math.pi / length * (sample_index + 0.5) * frequency_index
    )
    matrix[0] *= math.sqrt(1.0 / length)
    matrix[1:] *= math.sqrt(2.0 / length)
    return matrix


def merge_once(sequence, pair, new_token):
    output = []
    index = 0
    while index < len(sequence):
        if (
            index + 1 < len(sequence)
            and (sequence[index], sequence[index + 1]) == pair
        ):
            output.append(new_token)
            index += 2
        else:
            output.append(sequence[index])
            index += 1
    return output


def train_bpe(corpus, merge_count=8):
    sequences = [list(sequence) for sequence in corpus]
    merges = []
    next_token = max(max(sequence) for sequence in sequences) + 1

    for _ in range(merge_count):
        counts = Counter()
        for sequence in sequences:
            counts.update(zip(sequence[:-1], sequence[1:]))
        if not counts:
            break
        pair, _ = counts.most_common(1)[0]
        merges.append((pair, next_token))
        sequences = [
            merge_once(sequence, pair, next_token) for sequence in sequences
        ]
        next_token += 1
    return merges


def bpe_encode(sequence, merges):
    encoded = list(sequence)
    for pair, token in merges:
        encoded = merge_once(encoded, pair, token)
    return encoded


def bpe_decode(sequence, merges):
    reverse_table = {token: pair for pair, token in merges}

    def expand(token):
        if token not in reverse_table:
            return [token]
        left, right = reverse_table[token]
        return expand(left) + expand(right)

    decoded = []
    for token in sequence:
        decoded.extend(expand(token))
    return decoded


action_chunk = torch.tensor(
    [
        [0.1, -0.2],
        [0.2, -0.1],
        [0.4, 0.1],
        [0.3, 0.2],
        [0.0, 0.1],
        [-0.1, -0.1],
        [-0.2, -0.2],
        [0.0, -0.1],
    ],
    device=device,
)
dct = dct_matrix(action_chunk.size(0), device)
coefficients = dct @ action_chunk
quantized = torch.round(coefficients * 16).to(torch.int64)
minimum_value = int(quantized.min().item())
base_tokens = (quantized - minimum_value).flatten().tolist()
merges = train_bpe([base_tokens, base_tokens, base_tokens[::-1]])
encoded = bpe_encode(base_tokens, merges)
decoded = bpe_decode(encoded, merges)
decoded_quantized = (
    torch.tensor(decoded, device=device, dtype=torch.int64).view_as(quantized)
    + minimum_value
)
reconstructed = dct.T @ (decoded_quantized.float() / 16.0)

print("FAST base tokens:", len(base_tokens))
print("FAST BPE tokens:", len(encoded))
print("BPE round-trip exact:", torch.equal(decoded_quantized, quantized))
print("DCT reconstruction MSE:", F.mse_loss(reconstructed, action_chunk).item())


## References and provenance

- ACT official implementation: ResNet-18, 4-layer CVAE encoder, 4-layer observation encoder, 7-layer decoder, learned action queries, L1 + KL training.
- Diffusion Policy official `ConditionalUnet1D`: three down widths, two conditioned residual blocks per stage, two middle blocks, mirrored up path.
- Physical Intelligence openpi pi0: SigLIP So400m/14 image encoder (So400m depth 27), Gemma/PaliGemma + 18-layer action expert, GQA/RoPE, block attention mask, state/action/time suffix, flow target `noise-actions`.
- FAST: frequency-space action representation followed by discrete BPE compression.
